In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

In [23]:
df = pd.read_csv('../../datasets/c_text_youm7/youm7_nlp_preprocessed_data.csv')

In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12089 entries, 0 to 12088
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   category_label   12089 non-null  str  
 1   summary_no_stem  12089 non-null  str  
 2   summary_stemmed  12089 non-null  str  
dtypes: str(3)
memory usage: 5.2 MB


# Building Logistic Regression

# Stemmed Training

In [25]:
df_stemmed = df[['category_label', 'summary_stemmed']].copy()

# Testing valid across both stemming, non stemmed

In [26]:
def get_classification_results(
    y_true,
    y_pred,
    decision_scores,
    classes
):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    decision_scores = np.asarray(decision_scores)

    # Classification report
    report = classification_report(
        y_true,
        y_pred,
        labels=classes,
        output_dict=True,
        zero_division=0
    )

    # Confusion matrix
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=classes
    )

    rows = []

    # Per-class metrics
    for i, label in enumerate(classes):

        metrics = report[str(label)]

        # One-vs-Rest binary classification
        y_binary = (y_true == label).astype(int)

        # True Positives, False Positives,
        # True Negatives, False Negatives
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)

        # Specificity
        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0 else np.nan
        )

        # ROC-AUC and PR-AUC
        if len(np.unique(y_binary)) == 2:

            roc_auc = roc_auc_score(
                y_binary,
                decision_scores[:, i]
            )

            pr_auc = average_precision_score(
                y_binary,
                decision_scores[:, i]
            )

        else:
            roc_auc = np.nan
            pr_auc = np.nan

        rows.append({
            "Category": label,
            "Precision": metrics["precision"],
            "Recall": metrics["recall"],
            "F1-score": metrics["f1-score"],
            "Specificity": specificity,
            "ROC-AUC": roc_auc,
            "PR-AUC (AP)": pr_auc,
            "Accuracy": np.nan,
            "Support": int(metrics["support"])
        })

    # Create per-class DataFrame
    results_df = pd.DataFrame(rows)

    # Macro Average
    macro_row = {
        "Category": "Macro Average",
        "Precision": report["macro avg"]["precision"],
        "Recall": report["macro avg"]["recall"],
        "F1-score": report["macro avg"]["f1-score"],
        "Specificity": results_df["Specificity"].mean(),
        "ROC-AUC": results_df["ROC-AUC"].mean(),
        "PR-AUC (AP)": results_df["PR-AUC (AP)"].mean(),
        "Accuracy": np.nan,
        "Support": len(y_true)
    }

    # Weighted Average
    weights = results_df["Support"].to_numpy()

    weighted_row = {
        "Category": "Weighted Average",
        "Precision": report["weighted avg"]["precision"],
        "Recall": report["weighted avg"]["recall"],
        "F1-score": report["weighted avg"]["f1-score"],
        "Specificity": np.average(
            results_df["Specificity"],
            weights=weights
        ),
        "ROC-AUC": np.average(
            results_df["ROC-AUC"],
            weights=weights
        ),
        "PR-AUC (AP)": np.average(
            results_df["PR-AUC (AP)"],
            weights=weights
        ),
        "Accuracy": np.nan,
        "Support": len(y_true)
    }

    # Overall Accuracy
    accuracy_row = {
        "Category": "Overall Accuracy",
        "Precision": np.nan,
        "Recall": np.nan,
        "F1-score": np.nan,
        "Specificity": np.nan,
        "ROC-AUC": np.nan,
        "PR-AUC (AP)": np.nan,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Support": len(y_true)
    }

    # Combine all rows
    results_df = pd.concat(
        [
            results_df,
            pd.DataFrame([
                macro_row,
                weighted_row,
                accuracy_row
            ])
        ],
        ignore_index=True
    )

    return results_df

# Testing the Stemmed Logistic Regression

# No Stem Training

In [27]:
df_no_stem = df[['category_label', 'summary_no_stem']].copy()

In [28]:
# 70% Train, 30% Temp (Dev + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    df_no_stem["summary_no_stem"],
    df_no_stem["category_label"],
    test_size=0.30,
    random_state=42,
    stratify=df_no_stem["category_label"]
)

# 15% Dev, 15% Test
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [29]:
from sklearn.linear_model import LogisticRegression

In [30]:
C_values = [0.001, 0.01, 0.1, 1, 2, 5, 10, 20, 50, 100]

best_c = None
best_dev_f1 = -1

results = []

for c in C_values:

    candidate = make_pipeline(
        TfidfVectorizer(
            ngram_range=(1, 2),
            norm="l2",
            sublinear_tf=True
        ),

        LogisticRegression(
            C=c,
            solver="lbfgs",
            max_iter=3000,
            random_state=42
        )
    )

    candidate.fit(X_train, y_train)

    # Train evaluation
    train_preds = candidate.predict(X_train)

    train_f1 = f1_score(
        y_train,
        train_preds,
        average="macro"
    )

    # Dev evaluation
    dev_preds = candidate.predict(X_dev)

    dev_f1 = f1_score(
        y_dev,
        dev_preds,
        average="macro"
    )

    dev_acc = accuracy_score(y_dev, dev_preds)

    results.append({
        "C": c,
        "Train Macro F1": train_f1,
        "Dev Macro F1": dev_f1,
        "Dev Accuracy": dev_acc
    })

    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        best_c = c


results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

print(f"\nBest C: {best_c}")
print(f"Best Dev Macro F1: {best_dev_f1:.4f}")

      C  Train Macro F1  Dev Macro F1  Dev Accuracy
  0.001        0.038428      0.038459      0.155543
  0.010        0.558228      0.608044      0.594595
  0.100        0.899698      0.868074      0.867071
  1.000        0.982873      0.905287      0.905130
  2.000        0.994768      0.911981      0.911748
  5.000        0.998665      0.918579      0.918367
 10.000        0.999874      0.920853      0.920574
 20.000        1.000000      0.920212      0.920022
 50.000        0.999874      0.921267      0.921125
100.000        1.000000      0.923439      0.923331

Best C: 100
Best Dev Macro F1: 0.9234


In [31]:
X_train_final = pd.concat([X_train, X_dev])
y_train_final = pd.concat([y_train, y_dev])

In [32]:
lr_no_stem_model = make_pipeline(
    TfidfVectorizer(
        ngram_range=(1, 2),
        norm="l2",
        sublinear_tf=True
    ),

    LogisticRegression(
        C=best_c,
        solver="lbfgs",
        max_iter=3000,
        random_state=42
    )
)

In [33]:
lr_no_stem_model.fit(
    X_train_final,
    y_train_final
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](7,)","['اقتصاد_وبورصة','حوادث_وقضايا','رياضة_مصرية',...,'علوم_وتكنولوجيا', 'فن_وثقافة','محافظات_ومجتمع']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [35]:
test_preds = lr_no_stem_model.predict(X_test)

print(
    f"Test Accuracy: "
    f"{accuracy_score(y_test, test_preds):.4f}"
)

print(
    f"Test Macro F1: "
    f"{f1_score(y_test, test_preds, average='macro'):.4f}"
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        test_preds,
        zero_division=0
    )
)

Test Accuracy: 0.9388
Test Macro F1: 0.9390

Classification Report:
                  precision    recall  f1-score   support

   اقتصاد_وبورصة       0.95      0.94      0.94       261
    حوادث_وقضايا       0.93      0.95      0.94       223
     رياضة_مصرية       0.99      0.99      0.99       252
سياسة_وأخبار_مصر       0.95      0.87      0.91       268
 علوم_وتكنولوجيا       0.91      0.98      0.94       282
       فن_وثقافة       0.95      0.92      0.94       262
  محافظات_ومجتمع       0.91      0.92      0.92       266

        accuracy                           0.94      1814
       macro avg       0.94      0.94      0.94      1814
    weighted avg       0.94      0.94      0.94      1814



# Testing No Stem Logistic Regression

In [39]:
lr_no_stem_test_preds = lr_no_stem_model.predict(X_test)

lr_no_stem_test_scores = (
    lr_no_stem_model.decision_function(X_test)
)

lr_no_stem_results = get_classification_results(
    y_true=y_test,
    y_pred=lr_no_stem_test_preds,
    decision_scores=lr_no_stem_test_scores,
    classes=lr_no_stem_model.classes_
)

In [40]:
display(
    lr_no_stem_results.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}",
        "Specificity": "{:.4f}",
        "ROC-AUC": "{:.4f}",
        "PR-AUC (AP)": "{:.4f}",
        "Accuracy": "{:.4f}",
        "Support": "{:,.0f}"
    }, na_rep="—")
)

,Category,Precision,Recall,F1-score,Specificity,ROC-AUC,PR-AUC (AP),Accuracy,Support
0,اقتصاد_وبورصة,0.9496,0.9387,0.9441,0.9916,0.9951,0.9796,—,261
1,حوادث_وقضايا,0.9258,0.9507,0.9381,0.9893,0.9959,0.9647,—,223
2,رياضة_مصرية,0.9881,0.9921,0.9901,0.9981,0.9995,0.9968,—,252
3,سياسة_وأخبار_مصر,0.9472,0.8694,0.9066,0.9916,0.9789,0.9375,—,268
4,علوم_وتكنولوجيا,0.9079,0.9787,0.9420,0.9817,0.9928,0.9672,—,282
5,فن_وثقافة,0.9490,0.9237,0.9362,0.9916,0.9934,0.9759,—,262
6,محافظات_ومجتمع,0.9108,0.9211,0.9159,0.9845,0.9891,0.9393,—,266
7,Macro Average,0.9398,0.9392,0.9390,0.9898,0.9921,0.9659,—,"1,814"
8,Weighted Average,0.9394,0.9388,0.9386,0.9896,0.9919,0.9656,—,"1,814"
9,Overall Accuracy,—,—,—,—,—,—,0.9388,"1,814"
